# Pass@K Evaluation on Math Problem with Deepseek Math 7B Model
## Code structure
## 1. Environment Setup
## 2. Import Library
## 3. Global Configuration of inference parameters
## 4. Initialize model with vLLM framework
## 5. Building CoT Prompt for inference
## 6. Extract answers functions
## 7. Pass@k Function
## 8. Download the dataset
## 9. Inference loops
## 10. Building the json file to store the results

## 1. Install neccessary library

In [ ]:
# Solve the conflict with vLLM and Cuda
!pip install "jedi>=0.16"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.7 MB/s eta 0:00:00


In [ ]:
# Install neccessary library for downloading model and dataset, acclerate framework
!pip -q install -U vllm datasets transformers accelerate
# Ignore the warning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.

## 2. Import the libraries

In [ ]:
import re, os, sys
import math
import json
from typing import List, Dict, Any

from datasets import load_dataset
from vllm import LLM, SamplingParams

## 3. Global Configuration
1. choose dataset for DATASET varaible
2. choose k value for pass@k
3. choose the name for results file

In [ ]:
MODEL_ID = "deepseek-ai/deepseek-math-7b-instruct"
DATASET = "gsm8k" #"HuggingFaceH4/aime_2024", "HuggingFaceH4/MATH-500"
SPLIT = "test"
# k value of pass@k
K_LIST = [1, 4, 8, 16, 32, 64, 128]
N_SAMPLES = max(K_LIST)
# Generate more likely tokens
TEMPERATURE = 0.6
TOP_P = 0.95
# Answer in 512 tokens
MAX_NEW_TOKENS = 512
BATCH_SIZE = 16
TENSOR_PARALLEL_SIZE = 1
# The name of results json file
RESULT_DIR = "deepseek_SFT_passk_results_MATH.json"

## 4. Downloading model with vLLM framework to accerlerate inference

In [ ]:
# Patch the ipykernel file on disk so spawned worker processes inherit the fix (Gemini fix)
os.system("sed -i 's/raise io.UnsupportedOperation(\"fileno\")/return 1/g' /usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py")

# In-memory patch for the current process
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

# Initialize the model
llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    trust_remote_code=True,
    max_model_len=2048,
    gpu_memory_utilization=0.90,
    enable_lora=True,
)

# Samping parameters for vLLM
sampling_params = SamplingParams(
    n=N_SAMPLES,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_NEW_TOKENS,
)

INFO 05-05 16:49:11 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'enable_lora': True, 'model': 'deepseek-ai/deepseek-math-7b-instruct'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

INFO 05-05 16:49:30 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-05 16:49:30 [nixl_utils.py:34] NIXL is not available
WARNING 05-05 16:49:30 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-05 16:49:30 [model.py:555] Resolved architecture: LlamaForCausalLM
INFO 05-05 16:49:30 [model.py:1680] Using max model len 2048
INFO 05-05 16:49:30 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-05 16:49:30 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-05 16:49:30 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

(EngineCore pid=1878) INFO 05-05 16:49:32 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='deepseek-ai/deepseek-math-7b-instruct', speculative_config=None, tokenizer='deepseek-ai/deepseek-math-7b-instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_v

Loading pt checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=1878) INFO 05-05 16:50:21 [default_loader.py:384] Loading weights took 11.21 seconds
(EngineCore pid=1878) INFO 05-05 16:50:21 [punica_selector.py:20] Using PunicaWrapperGPU.
(EngineCore pid=1878) INFO 05-05 16:50:22 [gpu_model_runner.py:4879] Model loading took 12.95 GiB memory and 47.785173 seconds
(EngineCore pid=1878) INFO 05-05 16:50:37 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/dfaa09e531/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=1878) INFO 05-05 16:50:37 [backends.py:1128] Dynamo bytecode transform time: 14.82 s
(EngineCore pid=1878) INFO 05-05 16:50:43 [backends.py:376] Cache the graph of compile range (1, 8192) for later use
(EngineCore pid=1878) INFO 05-05 16:50:51 [backends.py:391] Compiling a graph for compile range (1, 8192) takes 13.16 s
(EngineCore pid=1878) INFO 05-05 16:50:58 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/b44620864b086753c2f

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 102/102 [00:19<00:00,  5.23it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 70/70 [00:06<00:00, 11.15it/s]


(EngineCore pid=1878) INFO 05-05 16:51:42 [gpu_model_runner.py:6133] Graph capturing finished in 27 secs, took 1.05 GiB
(EngineCore pid=1878) INFO 05-05 16:51:42 [gpu_worker.py:599] CUDA graph pool memory: 1.05 GiB (actual), 0.95 GiB (estimated), difference: 0.1 GiB (9.6%).
(EngineCore pid=1878) INFO 05-05 16:51:42 [core.py:299] init engine (profile, create kv cache, warmup model) took 80.34 s (compilation: 35.92 s)
(EngineCore pid=1878) INFO 05-05 16:51:43 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])


## 5. Building CoT Prompt

In [ ]:
def build_prompt(question):
    return (
        "<|im_start|>system\n"
        "You are a helpful assistant.<|im_end|>\n"
        "<|im_start|>user\n"
        f"{question}\n"
        "Please reason step by step, and put your final answer within\\boxed{{}}.<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

## 6. Extract answers Functions

In [ ]:
# re pattern
_gold_re = re.compile(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)")

def normalize_number_str(s):
  '''normalize the answer into str'''
    s = s.strip().replace(",", "")
    if re.fullmatch(r"[-+]?\d+\.0+", s):
        s = s.split(".")[0]
    return s

def extract_gold_answer(answer_text):
  '''Extract truth answers'''
    m = _gold_re.search(answer_text)
    if m:
        return normalize_number_str(m.group(1))
    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", answer_text)
    return normalize_number_str(nums[-1]) if nums else ""

def extract_pred_answer(completion):
  '''Extarct answers from generation.'''
    m = re.search(r"\\boxed\{([^}]*)\}", completion)
    if m:
        inside = m.group(1)
        nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", inside)
        if nums:
            return normalize_number_str(nums[-1])

    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", completion)
    if nums:
        return normalize_number_str(nums[-1])

    return ""

## 7. Unbiased pass@k estimation function

In [ ]:
def estimate_pass_at_k(n: int, c: int, k: int) -> float:
    """
    Unbiased estimator:
    pass@k = 1 - C(n-c, k) / C(n, k)
    """
    if c <= 0:
        return 0.0
    if n - c < k:
        return 1.0
    def log_comb(a: int, b: int) -> float:
        return math.lgamma(a + 1) - math.lgamma(b + 1) - math.lgamma(a - b + 1)

    log_num = log_comb(n - c, k)
    log_den = log_comb(n, k)
    ratio = math.exp(log_num - log_den)
    return 1.0 - ratio

## 8. Download the dataset
1. If you are testing GSM8K, use the first line.
2. If you are testting MATH500 or AIME, comment first line and uncomment second line

In [ ]:
ds = load_dataset("gsm8k", "main", split="test")
# ds = load_dataset(DATASET, split="test")

print(f"Loaded {len(examples)} {DATASET} examples.")

README.md: 0.00B [00:00, ?B/s]

algebra/train-00000-of-00001.parquet:   0%|          | 0.00/505k [00:00<?, ?B/s]

algebra/test-00000-of-00001.parquet:   0%|          | 0.00/353k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1744 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1187 [00:00<?, ? examples/s]

...

precalculus/train-00000-of-00001.parquet:   0%|          | 0.00/354k [00:00<?, ?B/s]

precalculus/test-00000-of-00001.parquet:   0%|          | 0.00/242k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/746 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/546 [00:00<?, ? examples/s]

Loaded 490 MATH 500 examples.


## 9. Inference loops
1. Generate multiple outputs paralle
2. For each output, compare it with the ground truth
3. Record the question, generated answer and correctness
4. Caluculate the pass@k

In [ ]:
all_results: List[Dict[str, Any]] = []

for start in range(0, len(examples), BATCH_SIZE):
    # Build prompts
    batch = examples[start:start + BATCH_SIZE]
    prompts = [x["prompt"] for x in batch]
    # Generate outputs
    outputs = llm.generate(prompts, sampling_params)
    # Iterate each answer in a batch
    for ex, out in zip(batch, outputs):
        preds = []
        correct_flags = []
        # Extract answer from generated output and compare with ground truth
        for cand in out.outputs:
            text = cand.text
            pred = extract_pred_answer(text)
            is_correct = (pred != "" and pred == ex["gold"])
            # Record the details
            preds.append({
                "text": text,
                "pred": pred,
                "correct": is_correct,
            })
            correct_flags.append(is_correct)

        c = sum(correct_flags)

        row = {
            "question": ex["question"],
            "gold": ex["gold"],
            "num_correct": c,
            "samples": preds,
        }
        # Calculate pass@k value
        for k in K_LIST:
            row[f"pass@{k}"] = estimate_pass_at_k(N_SAMPLES, c, k)

        all_results.append(row)

    print(f"Processed {min(start + BATCH_SIZE, len(examples))}/{len(examples)}")

Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2048 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Processed 16/490


Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2048 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

...

Processed prompts:   0%|          | 0/2048 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Processed 480/490


Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1280 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Processed 490/490


## 10. Store the pass@k result and samples in json file and export it.

In [ ]:
# Calculate average of pass@k value
metrics = {}
for k in K_LIST:
    metrics[f"pass@{k}"] = sum(r[f"pass@{k}"] for r in all_results) / len(all_results)

print("\n=== Final Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# Store and build a json file
with open(RESULT_DIR, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": MODEL_ID,
            "dataset": DATASET,
            "num_examples": len(all_results),
            "n_samples": N_SAMPLES,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_new_tokens": MAX_NEW_TOKENS,
            "metrics": metrics,
            "results": all_results,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"\nSaved detailed results to {RESULT_DIR}")


=== Final Metrics ===
pass@1: 0.2539
pass@4: 0.4218
pass@8: 0.5124
pass@16: 0.6035
pass@32: 0.6885
pass@64: 0.7612
pass@128: 0.8184

Saved detailed results to deepseek_SFT_passk_results.json
